## Decorators

**Decorators** modify or enhance functions/classes without changing their code. They wrap a function with additional behavior.

**Basic Syntax:**
```python
@decorator
def function():
    pass

# Equivalent to: function = decorator(function)
```


In [3]:
def decorator_function(original_function):
    # Wrapper function that calls the original function
    def wrapper_function():
        return original_function()
    # Return the wrapper instead of calling it immediately
    return wrapper_function

def display():
    # Original function
    print('this is display function')

# Pass display into the decorator and store the returned wrapper
decorated_display = decorator_function(display)

# Prints the wrapper function object
print(decorated_display)

# Calls wrapper_function, which then calls display()
decorated_display()

<function decorator_function.<locals>.wrapper_function at 0x1048a93a0>
this is display function


In [7]:
def decorator_function(original_function):
    # Wrapper function that calls the original function
    def wrapper_function():
        return original_function()
    # Return the wrapper instead of calling it immediately
    return wrapper_function

@decorator_function
def display():
    # Original function
    print('this is display function')

display()   # this is same as saying display = decorator_function(display)

# suppose if we do this then it will throw error because right now wrapper is not accepting arguments but we are passing arguments
@decorator_function
def display_info(name, age):
    print("display_info ran with arguments ({}, {})".format(name, age))

display_info("ram", 22)

this is display function


TypeError: decorator_function.<locals>.wrapper_function() takes 0 positional arguments but 2 were given

## `*args` and `**kwargs`

**`*args`** - Captures variable number of **positional arguments** as a tuple
```python
def sum_all(*args):
    return sum(args)

sum_all(1, 2, 3)  # args = (1, 2, 3) → returns 6
```

**`**kwargs`** - Captures variable number of **keyword arguments** as a dictionary
```python
def print_info(**kwargs):
    for key, value in kwargs.items():
        print(f"{key}: {value}")

print_info(name="Alice", age=30)  # kwargs = {'name': 'Alice', 'age': 30}
```

**Together:**
```python
def function(required, *args, **kwargs):
    # required: normal parameter
    # args: tuple of extra positional args
    # kwargs: dict of extra keyword args
    pass
```

In [11]:
# Decorator function: takes the original function as input
def decorator_function(original_function):

    # Wrapper function replaces the original decorated function
    # *args collects positional arguments
    # **kwargs collects keyword/named arguments
    # This makes the decorator work for functions with any parameters
    def wrapper_function(*args, **kwargs):
        # Extra code added before the original function runs
        print("Wrapper executed this before {}".format(original_function.__name__))

        # Pass the same received arguments to the original function
        return original_function(*args, **kwargs)

    # Return the wrapper function
    return wrapper_function


@decorator_function
def display():
    # Original function with no arguments
    print('this is display function')


# Same as: display = decorator_function(display)
# Calling display() actually calls wrapper_function()
display()


@decorator_function
def display_info(name, age):
    # Original function with arguments
    print("display_info ran with arguments ({}, {})".format(name, age))


# Same as: display_info = decorator_function(display_info)
# "ram" goes into *args, then wrapper passes it to original_function
display_info("ram", 22)

Wrapper executed this before display
this is display function
Wrapper executed this before display_info
display_info ran with arguments (ram, 22)


In [12]:
## we can also use decorator class instead of decorator functions

class decorator_class(object):
    def __init__(self, original_function):
        self.original_function = original_function

    def __call__(self, *args, **kwargs):
        print("Call method executed this before: {}".format(self.original_function))
        return self.original_function(*args, **kwargs)
    
@decorator_class
def display():
    print('this is display function')

display()

@decorator_class
def display_info(name, age):
    print("display_info ran with arguments ({}, {})".format(name, age))

display_info("ram", 22)

Call method executed this before: <function display at 0x104eb3880>
this is display function
Call method executed this before: <function display_info at 0x104ec1d00>
display_info ran with arguments (ram, 22)


### Solve the questions below


In [1]:
# 1. Write a decorator: timer
#    measures how long a function takes to run
#    prints: "function_name ran in 0.0023s"

# 2. Write a decorator: logger
#    prints before and after a function runs:
#    "Calling deposit..."
#    "deposit returned 1200"

# 3. Write a decorator: validate_positive
#    raises ValueError if any argument
#    passed to the function is negative


# PART 2 — Decorators with arguments

# 4. Write a decorator factory: repeat(n)
#    runs the decorated function n times
#    e.g. @repeat(3) runs the function 3 times


# PART 3 — Connect to BankAccount
# Apply your decorators to BankAccount methods:
# - @timer on get_balance()
# - @logger on deposit()
# - @validate_positive on deposit() and withdraw()

In [4]:
import time
from functools import wraps

# PART 1 — Basic decorators
# 1. timer decorator
def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} ran in {end - start:.4f}s")
        return result
    return wrapper

# 2. logger decorator
def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

# 3. validate_positive decorator
def validate_positive(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args[1:]:  # skip self because python calls deposit like (acc, amount), and the first index 0 is the self i.e object
            if isinstance(arg, (int, float)) and arg < 0:
                raise ValueError("Negative values are not allowed")
            
        for value in kwargs.values():
            if isinstance(arg, (int, float)) and value < 0:
                raise ValueError("Negative values are not allowed")
            
        return func(*args, **kwargs)
    return wrapper


# PART 2 — Decorator factory
# 4. repeat(n)
def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


class BankAccount:

    def __init__(self, name, balance):
        # Instance variables (unique for each object)
        self.name = name
        self.balance = balance

    @logger
    @validate_positive
    def deposit(self, amount):
        self.balance += amount
        return self.balance

    @validate_positive
    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            return self.balance
        else:
            return("Insufficient funds")
    
    @timer
    def get_balance(self):
        # Returns current balance
        return self.balance



In [6]:

# ============================================
# COMPREHENSIVE TESTS
# ============================================

print("=" * 60)
print("TESTING DECORATORS")
print("=" * 60)

# Test 1: deposit with logger
print("\n1. Testing deposit(500) - should show logger output:")
acc = BankAccount("Alice", 1000)
acc.deposit(500)

# Test 2: get_balance with timer
print("\n2. Testing get_balance() - should show timer:")
balance = acc.get_balance()
print(f"Balance: {balance}")

# Test 3: Negative deposit - should raise error
print("\n3. Testing deposit(-100) - should raise ValueError:")
try:
    acc.deposit(-100)
except ValueError as e:
    print(f"✓ Error caught: {e}")

# Test 4: Negative withdrawal - should raise error
print("\n4. Testing withdraw(-50) - should raise ValueError:")
try:
    acc.withdraw(-50)
except ValueError as e:
    print(f"✓ Error caught: {e}")

# Test 5: Valid withdrawal
print("\n5. Testing withdraw(200):")
new_balance = acc.withdraw(200)
print(f"New balance: {new_balance}")

# Test 6: Insufficient funds
print("\n6. Testing withdraw(5000) - insufficient balance:")
try:
    acc.withdraw(5000)
except ValueError as e:
    print(f"✓ Error caught: {e}")

# Test 7: repeat decorator
print("\n7. Testing @repeat(3) decorator:")
@repeat(3)
def greet(name):
    print(f"Hello {name}")

greet("Anish")

# Test 8: repeat with different number
print("\n8. Testing @repeat(5):")
@repeat(5)
def say_hi():
    print("Hi!")

say_hi()

# Test 9: Combining multiple decorators
print("\n9. Testing multiple decorators together:")
@timer
@logger
def calculate(x, y):
    time.sleep(0.1)  # Simulate some work
    return x + y

result = calculate(10, 20)
print(f"Final result: {result}")

# Test 10: Understanding decorator order
print("\n10. Understanding decorator order:")
print("When decorators are stacked:")
print("@logger")
print("@validate_positive")
print("def deposit(...):")
print("\nExecution order: validate_positive runs FIRST, then logger")
print("(Bottom decorator wraps first, top decorator wraps last)")

# Test 11: Creating another account
print("\n11. Creating a new account:")
acc2 = BankAccount("Bob", 500)
acc2.deposit(250)
print(f"Bob's balance: {acc2.get_balance()}")

print("\n" + "=" * 60)
print("ALL TESTS COMPLETED! ✓")
print("=" * 60)


# ============================================
# BONUS: Understanding decorator order visually
# ============================================

print("\n" + "=" * 60)
print("BONUS: DECORATOR ORDER EXPLANATION")
print("=" * 60)

print("""
When you write:
    @logger
    @validate_positive
    def deposit(self, amount):
        ...

It's the same as:
    deposit = logger(validate_positive(deposit))

So when you call deposit(100):
1. logger wrapper runs first (prints "Calling deposit...")
2. Inside logger, it calls validate_positive wrapper
3. validate_positive checks if 100 is positive ✓
4. validate_positive calls the original deposit function
5. Original deposit runs and returns new balance
6. validate_positive returns the result
7. logger prints "deposit returned 1500"
8. logger returns the result

""")

TESTING DECORATORS

1. Testing deposit(500) - should show logger output:
Calling deposit
deposit returned 1500

2. Testing get_balance() - should show timer:
get_balance ran in 0.0000s
Balance: 1500

3. Testing deposit(-100) - should raise ValueError:
Calling deposit
✓ Error caught: Negative values are not allowed

4. Testing withdraw(-50) - should raise ValueError:
✓ Error caught: Negative values are not allowed

5. Testing withdraw(200):
New balance: 1300

6. Testing withdraw(5000) - insufficient balance:

7. Testing @repeat(3) decorator:
Hello Anish
Hello Anish
Hello Anish

8. Testing @repeat(5):
Hi!
Hi!
Hi!
Hi!
Hi!

9. Testing multiple decorators together:
Calling calculate
calculate returned 30
calculate ran in 0.1027s
Final result: 30

10. Understanding decorator order:
When decorators are stacked:
@logger
@validate_positive
def deposit(...):

Execution order: validate_positive runs FIRST, then logger
(Bottom decorator wraps first, top decorator wraps last)

11. Creating a new ac